# Assignment 2: Regression and Classification Models
## Topic: Cholestrol and Heart Disease Predictions 

### Import Libraries 
Import necessary libraries for each step in this assignment. This includes libraries for data cleaning, preprocessing, visualization, modelling, evaluation metrics and hyperparameters. 

In [ ]:
#-----Data Manipulation-----
import numpy as np
import pandas as pd

#-----Data Visualization-----
import matplotlib.pyplot as plt
import seaborn as sns #Used in linear regression to visualize RMSE AND R^2 (sns.heatmap)

#-----Data Preprocessing----- 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

#-----Modeling: Regression and Classification----- 
from sklearn.linear_model import ElasticNet, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

#-----Hyperparameter Tuning-----
from sklearn.model_selection import GridSearchCV # Used to find best parameters

#-----Evaluation Metrics-----
from sklearn.metrics import (
    r2_score, mean_squared_error, #Used for regression evaluation 
    accuracy_score, precision_score, recall_score, f1_score, #Used for classification Evaluation 
    roc_auc_score, average_precision_score, # Used to calculate AUROC and AUPRC scores resepctively 
    roc_curve, precision_recall_curve, # Used to plot AUROC and AUPRC curves
    )

### Data Preparation: Data Cleaning and Feature Engineering 
This section includes checking and preparing the data that will be input into the model. This includes: 
- Dropping features with high missingness (more than 15% missing)
- Imputing missing data 
- One-hot encoding categorical features
- Creating specific datasets depending on the target: chol_data and disease_data - converts 'num' into binary classification target for heart disease predication 

In [ ]:
#--------Open Data and read data----------------------------
heart_data = pd.read_csv("../Data/heart_disease_uci(1).csv")
#------------------------------------------------------------

#-------------------------------------
#Examine Data structure 
#-------------------------------------
print(heart_data.info())
print(heart_data.head())
# print(heart_data.describe())
# print(heart_data.isnull().sum())
print(heart_data['num'].value_counts()) #check the structure of num before binary classification 

#-------------------------------------
#Create functions to clean datasets
#-------------------------------------
def dropMissing(data: pd.DataFrame, threshold = 0.15):  #This function creates a threshold to drop rows that have too much missingness 
    missing = data.isna().sum() / data.shape[0] > threshold
    toDrop = missing[missing].index
    print(f"Dropping {toDrop}") #tells what columns I am dropping
    data.drop(columns=toDrop, inplace=True)

def fillMissing(data: pd.DataFrame):  #This function is used to impute missing values for rows that still have some missing data
    missing = data.isna().sum() > 0
    for colName in missing[missing].index:
        if data[colName].dtype == 'object':  #for categorical data missing values are imputed with 'Unknown'
            data[colName] = data[colName].fillna("Unknown")
        else:
            data[colName] = data[colName].fillna(data[colName].median()) #imputes median for numerical data

#---------------------------------------------------
#Create cholestrol dataset for linear regression
#---------------------------------------------------
chol_data = heart_data.copy() #chol model dataset
chol_data = chol_data.dropna(subset=['chol']) # drop missing target rows
#----'chol' feature engineering----
dropMissing(chol_data, .50)
fillMissing(chol_data)
chol_data = pd.get_dummies(chol_data, drop_first=True) #one hot encode categorical/object features
chol_data.info()

#------------------------------------------------------
#Create heart disease model for logistic regression
#------------------------------------------------------
disease_data = heart_data.copy() #num model dataset
#----'num' feature engineering----
disease_data['num'] = disease_data['num'] > 0 # convert 'num' from multiclass (0-4) to binary boolean values (False (0 - no disease) and True (1-4- disease))
#print(disease_data['num'].unique()) #checks to see if properly converted to True and False
dropMissing(disease_data, .50)
fillMissing(disease_data)
disease_data = pd.get_dummies(disease_data, drop_first=True) #one hot encode categorical/object features

### Linear Regression Model: ElasticNet 
#### For Linear Regression: Predict Cholestrol Levels (column name = 'chol')
In this section, I implement a linear regression model using ElasticNet to predict an individual's cholestrol level based on other health features. The model is evaluated using RMSE and R^2 performance metrics. To identify the most optimized configurations of the model, results for alpha and l1_ratio combinations are visualzied with heatmap to view the models performance across different values. 

In [ ]:
#-------------------------------------
#Model Specific Data Preprocessing 
#-------------------------------------
# Regression target: 'chol' -> Define (X) features and target variable (y) 
X = chol_data.drop(columns=['chol', 'num'])  # Drop targets from features 
y = chol_data['chol']  # Cholesterol level as target for regression

#  Split the data into training and testing sets: 70% training and 30% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

#-------------------------------------
#Create a Hyperparameter grid 
#-------------------------------------
rmseArray = np.zeros([11, 11])  #arrays used to store scores of alpha and l1_ratio combinations
r2Array = np.zeros([11, 11])

best_rmse, best_r2 = float('inf'), 0 #best score tracker intialization
best_rmse_config = [None, None]
best_r2_config = [None, None]

alphas = np.logspace(-2, 1, num=11) #alpha values tested, spaced logarithmically between 0.01 and 10
l1_ratio = np.arange(0, 1.1, 0.1)   #l1_ratio values tested, between 0-1 in steps of 0.1

#------------------------------------------------------------------------------------------------------------
#GridSearch: Loop through combinations of alpha and l1_ration to find best RMSE AND R^2 Scores and model
#------------------------------------------------------------------------------------------------------------
for i, myAlpha in enumerate(alphas):
    for j, myL1 in enumerate(l1_ratio):
        
        #Train ElasticNet model
        model = ElasticNet(alpha=myAlpha, l1_ratio=myL1)  
        model.fit(X_train, y_train) #fit data to training data 

        # Predict on test data
        y_pred =  model.predict(X_test)
        
        # Evaluate the model
        rmse = np.sqrt(mean_squared_error(y_test, y_pred)) #calculate eval. metrics
        r2 = r2_score(y_test, y_pred)
        if rmse < best_rmse:   #Tracks best RMSE
            best_rmse = rmse
            best_rmse_config = [str(myAlpha), str(myL1)]
        if r2 > best_r2:       #Tracks best R^2
            best_r2 = r2
            best_r2_config = [str(myAlpha), str(myL1)]
        rmseArray[i][j] = rmse  #store results in array for heat map
        r2Array[i][j] = r2

#-------------------------------------
#Plot Heatmaps 
#-------------------------------------
sns.heatmap(rmseArray, xticklabels=np.round(l1_ratio, 1), yticklabels=np.round(alphas, 2))  
plt.title("RMSE Heatmap")
plt.xlabel("L1 Ratio")
plt.ylabel("Alpha")
plt.show()

sns.heatmap(r2Array, xticklabels=np.round(l1_ratio, 2), yticklabels=np.round(alphas, 2))
plt.title("R? Heatmap")
plt.xlabel("L1 Ratio")
plt.ylabel("Alpha")
plt.show()

#------------Print Best Scores --------------
print(f"Best RMSE: {best_rmse:.4f} at Alpha={best_rmse_config[0]}, L1 Ratio={best_rmse_config[1]}")
print(f"Best r2: {best_r2:.4f} at Alpha={best_r2_config[0]}, L1 Ratio={best_r2_config[1]}")

### Classification Models: Logisitic Regression vs. k-Nearest-Neighbours(k-NN)
#### For Classification: Predict presence of heart disease as binary classification task (column name = 'num')
In this part, I compared two classification models - Logisitic Regression and k-NN - to predict the presence of heart disease. The target variable  for heart disease ('num') was split into binary classification to indicate if heart diseas was present (True) or  not present (False). To determine which model performed best, I used evaluation metrics such as accuracy, F1_score, AUROC (Area Under the ROC Curve) and AUPRC (Area Under the Precision-Recall Curve). AOC and Precision-Recall curves were visualized alongside their scores to view each models predicability. 

In [ ]:
#-------------------------------------
#Model Specific Data Preprocessing 
#-------------------------------------
#Classifcation target: 'num' -> Define (X) features and target variable (y)
X = disease_data.drop(columns=['chol', 'num'])  # Drop targets from features 
y = disease_data['num']  # Cholesterol level as target for regression

#  Split the data into training and testing sets: 70% training and 30% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Standardize features
scaler = StandardScaler()
X_train= scaler.fit_transform(X_train)
X_test= scaler.transform(X_test)

#-------------------------------------
#Logistic Regression with GridSearchCV
#-------------------------------------
param_grid_log_reg = {'solver': ['liblinear', 'saga'],'penalty': ['l1', 'l2']} #hyperparameters to test 
#Cross-Validation of configurations 
grid_search_log_reg = GridSearchCV(                     
    LogisticRegression(random_state=42, max_iter=100, C=1.0), 
    param_grid_log_reg, 
    cv=5, 
    scoring='accuracy'
    ) 
grid_search_log_reg.fit(X_train, y_train) #fit to grid searcj model to training data 

#---------------------------------------
# k-Nearest Neighbours with GridSearchCV
#---------------------------------------
param_grid_knn = {'n_neighbors': [1,5,10],'metric': ['euclidean', 'manhattan', 'minkowski']} 
grid_search_knn = GridSearchCV(KNeighborsClassifier(), param_grid_knn, cv=5, scoring='accuracy')
grid_search_knn.fit(X_train, y_train)

#--------------------------------------------------
# Evaluate the Logisitic Regression and k-NN models
#--------------------------------------------------
models = {'Log-Reg': grid_search_log_reg, 'k-NN': grid_search_knn}
for name, model in models.items(): #loop through each model, generate predictions and calculate eval. metrics
    y_pred = model.predict(X_test)  #predicts class labels on test set
    accuracy = accuracy_score(y_test, y_pred)    #evaluation metrics accuracy, precision, recall, f1
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    print(f"{name} - Accuracy: {accuracy:.2f}, Precision: {precision:.2f}, Recall: {recall:.2f}, F1 Score: {f1:.2f}")

#Print best parameters and scores
print(f"Best parameters for LogRegression: {grid_search_log_reg.best_params_}")
print(f"Best score for LogRegression: {grid_search_log_reg.best_score_}")
print(f"Best parameters for k-NN: {grid_search_knn.best_params_}")
print(f"Best score for k-NN: {grid_search_knn.best_score_}")

#---------------------------------------------------------------
# Predict probabilities for ROC and precison-recall calculations 
#---------------------------------------------------------------
y_scores_knn = grid_search_knn.predict_proba(X_test)[:, 1]  #probability of postive class for knn
y_scores_log = grid_search_log_reg.predict_proba(X_test)[:, 1] #probability of postive class for log_reg

#----------------------------------------
# Compute ROC curve and AUROC/AUPRC scores
#-----------------------------------------
#compute ROC curve values: false postive rate (fpr) and true postive rate (tpr)
fpr_knn, tpr_knn, _ = roc_curve(y_test, y_scores_knn)
fpr_log, tpr_log, _ = roc_curve(y_test, y_scores_log)
#calculate AUROC scores
auroc_log = roc_auc_score(y_test, y_scores_log)
auroc_knn = roc_auc_score(y_test, y_scores_knn)
#calculate AUPRC scores
auprc_log = average_precision_score(y_test, y_scores_log)
auprc_knn = average_precision_score(y_test, y_scores_knn)

#----------------- Function to Plot both models: AUROC AND AUPRC ------------------
def plot_curves(tpr,fpr,auroc,precision,recall,auprc,model_name,minority_class=0.1):  #Function obtained from Lab 4
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))                             #makes 2 subplots side by side

    # Plot ROC curve
    ax1.plot(fpr, tpr, color='green', lw=2, label='AUROC = %0.2f' % auroc) #AUROC graph 
    ax1.plot([0, 1], [0, 1], color='red', lw=2, linestyle='--', label='Random') #diagonal line (random guess)
    ax1.set_xlim([0.0, 1.0])
    ax1.set_ylim([0.0, 1.05])
    ax1.set_xlabel('False Positive Rate')
    ax1.set_ylabel('True Positive Rate')
    ax1.set_title('{} ROC Curve'.format(model_name))
    ax1.legend(loc='lower right')

    # Plot Precision-Recall curve
    ax2.plot(recall, precision, color='purple', lw=2, label='AUPRC = %0.2f' % auprc) #AUPRC graph
    ax2.axhline(y=minority_class, color='red', lw=2, linestyle='--', label='Random') #random precision line based on minority class (postive class)
    ax2.set_xlabel('Recall')
    ax2.set_ylabel('Precision')
    ax2.set_title('{} Precision-Recall Curve'.format(model_name))
    ax2.legend(loc='lower left')

    plt.tight_layout()
    plt.show()

#-------------------------------------
# Compute Precision-Recall curve
#-------------------------------------
precision_knn, recall_knn, _ = precision_recall_curve(y_test, y_scores_knn)
plot_curves(tpr_knn, fpr_knn, auroc_knn, precision_knn, recall_knn, auprc_knn, model_name='k-NN') #call plotting curve function and inputs 

precision_log, recall_log, _ = precision_recall_curve(y_test, y_scores_log)
plot_curves( tpr_log, fpr_log, auroc_log, precision_log, recall_log, auprc_log, model_name='Logistic Regression')